# Pangolin — splice-site prediction & variant scoring

Pangolin is a SpliceAI-lineage deep-learning model that predicts tissue-specific splice-site probability from raw DNA and scores the splicing effect (gain/loss) of variants. This notebook demonstrates both registered tools: `run_pangolin_predict` for per-position splice-probability prediction and `run_pangolin_score_variants` for variant splice-effect scoring.

- Paper: https://doi.org/10.1186/s13059-022-02664-4
- Repository: https://github.com/tkzeng/Pangolin

In [1]:
# Public API for both Pangolin tools, plus the notebook docs helper.
from proto_tools.tools.rna_splicing.pangolin import (
    run_pangolin_predict,
    PangolinPredictInput,
    PangolinPredictConfig,
    run_pangolin_score_variants,
    PangolinScoreVariantsInput,
    PangolinScoreVariantsConfig,
    PangolinVariant,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [2]:
# Input schema for splice-site prediction: the DNA sequences to score.
display_api_reference("pangolin-predict", "input", "run_pangolin_predict")

**Input** — `PangolinPredictInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | DNA sequence(s) >= 10001 bp; predictions cover the central (len - 10000) positions |

In [3]:
# Config schema: which tissues to ensemble and which device to run on.
display_api_reference("pangolin-predict", "config", "run_pangolin_predict")

**Config** — `PangolinPredictConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int | None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `tissues` | `list[Literal['HEART', 'LIVER', 'BRAIN', 'TESTIS']]` | `['HEART', 'LIVER', 'BRAIN', 'TESTIS']` | Tissues whose splice predictions are ensembled (HEART, LIVER, BRAIN, TESTIS) |

In [4]:
# Output schema: per-sequence predictions with per-position, per-tissue scores.
display_api_reference("pangolin-predict", "output", "run_pangolin_predict")

**Output** — `PangolinPredictOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.rna_splicing.pangolin.pangolin_predict.PangolinPrediction]` | required | Per-sequence Pangolin predictions (1:1 with input sequences) |

## Splice-site prediction

Pangolin needs at least 10,001 bp of input: it uses 5,000 bp of flanking context on each side and scores the central `len(sequence) - 10000` positions. We build a reproducible random 10,100 bp sequence so the example runs without any external genome file.

In [5]:
import random

# Reproducible ~10,100 bp DNA window. Central 100 positions get scored
# (10100 - 2 * 5000 = 100), using 5,000 bp of flank on each side.
random.seed(0)
seq = "".join(random.choice("ACGT") for _ in range(10100))

# Run prediction on CPU so the notebook works without a GPU.
result = run_pangolin_predict(
    PangolinPredictInput(sequences=[seq]),
    PangolinPredictConfig(device="cpu"),
)

prediction = result.results[0]
print(f"success: {result.success}")
print(f"scored positions: {len(prediction.scores)}")
print(f"tissues: {prediction.tissues}")
print("first 3 rows of per-tissue scores:")
for row in prediction.scores[:3]:
    print("  ", [round(score, 4) for score in row])

Running run_pangolin_predict [00:00]

success: True
scored positions: 100
tissues: ['HEART', 'LIVER', 'BRAIN', 'TESTIS']
first 3 rows of per-tissue scores:
   [0.0472, 0.0499, 0.048, 0.0505]
   [0.049, 0.0517, 0.0508, 0.0517]
   [0.0466, 0.0501, 0.0494, 0.0517]


In [6]:
# Advanced: restrict the ensemble to a subset of tissues. Each score row then
# has one column per requested tissue, in the order given.
subset = run_pangolin_predict(
    PangolinPredictInput(sequences=[seq]),
    PangolinPredictConfig(device="cpu", tissues=["BRAIN", "HEART"]),
)

subset_prediction = subset.results[0]
print(f"tissues: {subset_prediction.tissues}")
print(f"score shape: {len(subset_prediction.scores)} positions x "
      f"{len(subset_prediction.scores[0])} tissues")

Running run_pangolin_predict [00:00]

tissues: ['BRAIN', 'HEART']
score shape: 100 positions x 2 tissues


## Variant splice-effect scoring

`run_pangolin_score_variants` compares the predicted splice-site probability between the reference and alternate sequence over a `± distance` window, reducing across the selected tissues to a per-position splice gain (largest increase) and loss (largest decrease). Each variant carries its own reference window — no genome FASTA is required.

In [7]:
# Reuse a ~10,200 bp window and place a single-nucleotide variant at the centre,
# leaving >= 5,000 bp of flank on each side (required by Pangolin).
random.seed(0)
seq = "".join(random.choice("ACGT") for _ in range(10200))

pos = 5100
ref = seq[pos]
alt = "A" if ref != "A" else "C"

variant = PangolinVariant(
    sequence=seq,
    variant_position=pos,
    reference_bases=ref,
    alternate_bases=alt,
)

# distance=50 reports splice scores 50 bp on each side of the variant.
result = run_pangolin_score_variants(
    PangolinScoreVariantsInput(variants=[variant]),
    PangolinScoreVariantsConfig(device="cpu", distance=50),
)

effect = result.results[0]
print(f"success: {result.success}")
print(f"largest increase: {effect.increase_score:.4f} at {effect.increase_position} bp")
print(f"largest decrease: {effect.decrease_score:.4f} at {effect.decrease_position} bp")
print(f"metrics: {effect.metrics}")

Running run_pangolin_score_variants [00:00]

success: True
largest increase: 0.0013 at -2 bp
largest decrease: -0.0013 at -1 bp
metrics: primary_metric='max_gain' max_gain=0.001259438693523407 max_loss=-0.0013429820537567139


In [8]:
# Schemas for the variant-scoring tool: config (tissues, distance, device) and output.
display_api_reference("pangolin-score-variants", "config", "run_pangolin_score_variants")
display_api_reference("pangolin-score-variants", "output", "run_pangolin_score_variants")

**Config** — `PangolinScoreVariantsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int | None` | `600` | Maximum execution time in seconds. None waits indefinitely. |
| `seed` | `int | None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `tissues` | `list[Literal['HEART', 'LIVER', 'BRAIN', 'TESTIS']]` | `['HEART', 'LIVER', 'BRAIN', 'TESTIS']` | Tissues whose splice predictions are ensembled (HEART, LIVER, BRAIN, TESTIS) |
| `distance` | `int` | `50` | bp on each side of the variant to report splice scores for |

**Output** — `PangolinScoreVariantsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.rna_splicing.pangolin.pangolin_score_variants.PangolinVariantEffect]` | required | Per-variant splice-effect scores (1:1 with input variants) |

In [9]:
# Export the variant-effect results. The default format is JSON; CSV is also
# supported via file_format="csv" (one row per variant with gain/loss summaries).
result.export("pangolin_variants")
print("Exported results to pangolin_variants.json (default JSON; pass file_format='csv' for CSV)")

Exported results to pangolin_variants.json (default JSON; pass file_format='csv' for CSV)
